# Clasificación de perros y gatos

Este notebook implementa únicamente los pasos solicitados: inspección visual, carga progresiva con `ImageDataGenerator`, RNA con EfficientNet-B0, entrenamiento con `ModelCheckpoint` y `EarlyStopping`, evaluación sobre test y guardado del mejor modelo.

In [ ]:
from pathlib import Path
import shutil
import random

import matplotlib.pyplot as plt
import numpy as np
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

ROOT = Path.cwd().parent if Path.cwd().name == "src" else Path.cwd()
DATASET_DIR = ROOT / "dogs-vs-cats" / "train"
SPLIT_DIR = ROOT / "dogs-vs-cats"
MODEL_PATH = ROOT / "models" / "dogs_vs_cats_best.keras"
IMAGE_SIZE = (200, 200)
BATCH_SIZE = 32

cat_files = sorted(DATASET_DIR.glob("cat.*.jpg"))
dog_files = sorted(DATASET_DIR.glob("dog.*.jpg"))
print(f"Gatos: {len(cat_files)} | Perros: {len(dog_files)}")

In [ ]:
def show_examples(files, title):
    fig, axes = plt.subplots(3, 3, figsize=(10, 10))
    for axis, file_path in zip(axes.flat, files[:9]):
        axis.imshow(load_img(file_path))
        axis.set_title(file_path.name)
        axis.axis("off")
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

show_examples(dog_files, "Nueve imágenes de perros")
show_examples(cat_files, "Nueve imágenes de gatos")

In [ ]:
def prepare_directory_split(test_size=0.2):
    """Create the directory structure required by flow_from_directory."""
    for split in ("train_split", "test_split"):
        for class_name in ("cat", "dog"):
            (SPLIT_DIR / split / class_name).mkdir(parents=True, exist_ok=True)

    rng = random.Random(SEED)
    for class_name, files in (("cat", cat_files), ("dog", dog_files)):
        shuffled = files.copy()
        rng.shuffle(shuffled)
        cutoff = int(len(shuffled) * (1 - test_size))
        for split, selected in (("train_split", shuffled[:cutoff]), ("test_split", shuffled[cutoff:])):
            destination = SPLIT_DIR / split / class_name
            for file_path in selected:
                target = destination / file_path.name
                if not target.exists():
                    shutil.copy2(file_path, target)

prepare_directory_split()

In [ ]:
train_datagen = ImageDataGenerator(rescale=1.0 / 255)
test_datagen = ImageDataGenerator(rescale=1.0 / 255)

trdata = train_datagen.flow_from_directory(
    SPLIT_DIR / "train_split",
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=True,
    seed=SEED,
)
tsdata = test_datagen.flow_from_directory(
    SPLIT_DIR / "test_split",
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False,
)
print("Clases:", trdata.class_indices)

In [ ]:
model = keras.Sequential([
    layers.Resizing(224, 224, input_shape=(*IMAGE_SIZE, 3)),
    EfficientNetB0(include_top=False, weights=None, input_shape=(224, 224, 3)),
    layers.GlobalAveragePooling2D(),
    layers.Dense(units=128, activation="relu"),
    layers.Dense(units=2, activation="softmax"),
])
model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
model.summary()

In [ ]:
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
checkpoint = ModelCheckpoint(MODEL_PATH, monitor="val_accuracy", save_best_only=True, mode="max")
early_stopping = EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)

history = model.fit(
    trdata,
    validation_data=tsdata,
    epochs=10,
    callbacks=[checkpoint, early_stopping],
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history["accuracy"], label="train")
axes[0].plot(history.history["val_accuracy"], label="test")
axes[0].set_title("Accuracy")
axes[0].legend()
axes[1].plot(history.history["loss"], label="train")
axes[1].plot(history.history["val_loss"], label="test")
axes[1].set_title("Loss")
axes[1].legend()
plt.tight_layout()
plt.show()

best_model = keras.models.load_model(MODEL_PATH)
test_loss, test_accuracy = best_model.evaluate(tsdata, verbose=1)
predictions = best_model.predict(tsdata, verbose=1)
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")
print("Predicciones generadas:", predictions.shape)